In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('ad_click_dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
import plotly.express as px
def plot_pie_chart(df, column_name):
    value_counts = df[column_name].value_counts(dropna=False)
    value_counts = value_counts.reset_index()
    value_counts.columns = [column_name, 'count']
    
    fig = px.pie(value_counts, names=column_name, values='count', 
                 title=f'{column_name} Distribution (Including Unknown)',
                 color_discrete_sequence=px.colors.qualitative.T10,
                 hole=0.4)
    
    fig.update_traces(textinfo='percent+label')

    fig.update_layout(title_x=0.5)

    fig.show()

categorical_columns = df.select_dtypes(include=['object']).columns
categorical_columns = categorical_columns.drop('full_name', errors='ignore')
for col in categorical_columns:
    plot_pie_chart(df, col)

In [ ]:
df_user_type = df.copy()
user_counts = df_user_type['full_name'].value_counts()

df_user_type['user_type'] = df_user_type['full_name'].apply(lambda x: 'First Time Users' if user_counts[x] == 1 else 'Recurring Users')
recurring_users = df_user_type[df_user_type.duplicated(subset=['full_name'], keep=False)]
first_time_users = df_user_type.drop_duplicates(subset=['full_name'], keep=False)

In [ ]:
recurring_users.describe(include='all')

In [ ]:
first_time_users.describe(include='all')

In [ ]:
feature_counts = df_user_type.groupby(['user_type', 'click']).size().reset_index(name='count')

feature_counts['click'] = feature_counts['click'].map({0: 'No Click', 1: 'Click'})

fig = px.bar(feature_counts,
             x='user_type', y='count', color='click',
             title='Ad Clicks by User Type (First Time vs Recurring)',
             labels={"user_type": "User Type", "count": "Number of Clicks", "click": "Click Status"},
             color_discrete_sequence=px.colors.qualitative.T10,
             text='count',
             barmode="group")

fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
fig.update_layout(
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    title_x=0.5,
    xaxis_title="User Type",
    yaxis_title="Number of Clicks",
    legend_title="Click Status",
    bargap=0.5,
    bargroupgap=0.1
)

fig.show()

In [ ]:
recurring_users.sort_values(by='full_name', ascending=True).head(30)

In [ ]:
df.update(df.groupby('full_name').transform(lambda x: x.ffill().bfill()))
df = df.drop(columns=['full_name'])

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col].fillna('Unknown', inplace=True)

In [ ]:
from sklearn.impute import KNNImputer
def knn_impute(df, n_neighbors=5):   
    df_encoded = df.copy()

    category_mappings = {}
    for col in df_encoded.select_dtypes(include='object').columns:
        df_encoded[col] = df_encoded[col].astype('category').cat.codes
        category_mappings[col] = dict(enumerate(df[col].astype('category').cat.categories))


    knn_imputer = KNNImputer(n_neighbors=n_neighbors)
    df_imputed = pd.DataFrame(knn_imputer.fit_transform(df_encoded), columns=df_encoded.columns)


    for col in df.select_dtypes(include='object').columns:
        df_imputed[col] = df_imputed[col].round().astype(int).map(category_mappings[col])

    return df_imputed

df_imputed = knn_impute(df, n_neighbors=5)


df = df_imputed

In [ ]:
df.isnull().sum()  


In [ ]:
age_bins = [15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65]
age_labels = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64']
df_visualization = df.copy()
df_visualization['age_group'] = pd.cut(df_visualization['age'], bins=age_bins, labels=age_labels, right=False)


click_counts = df_visualization.groupby(['age_group', 'click']).size().reset_index(name='count')

click_counts['click'] = click_counts['click'].map({0: 'No Click', 1: 'Click'})

fig = px.bar(
    click_counts, 
    x='age_group', 
    y='count', 
    color='click', 
    title='Age Group Distribution of Clicks vs No Clicks',
    labels={'age_group': 'Age group', 'count': 'Count', 'click': 'Click Status'},
    text='count',
    color_discrete_sequence=px.colors.qualitative.T10,
    category_orders={'age_group': age_labels}
)

fig.update_layout(
    barmode='stack',
    xaxis_title='Age group',
    yaxis_title='Count',
    legend_title='Click Status',
    title_x=0.5
)

fig.show()

In [ ]:
feature_counts = df.groupby([df['click'].map({1: 'Click', 0: 'No Click'}), 'ad_position']).size().reset_index(name='count')

total_count = feature_counts['count'].sum()
feature_counts['percent'] = feature_counts['count'] / total_count * 100

fig = px.sunburst(feature_counts, path=['click', 'ad_position'], values='count',
                 color='count', color_continuous_scale='Teal',
                 title='Ad Clicks by Ad position')

fig.update_traces(
    textinfo='label+percent entry',  
    texttemplate='<b>%{label}</b><br>%{percentEntry:.2%}'
)

fig.update_layout(title_text='Ad Clicks by Ad position',
                  title_x=0.5, width=900, height=600)  

fig.show()


In [ ]:
feature_counts = df.groupby([df['click'].map({1: 'Click', 0: 'No Click'}), 'time_of_day']).size().reset_index(name='count')

total_count = feature_counts['count'].sum()
feature_counts['percent'] = feature_counts['count'] / total_count * 100

fig = px.sunburst(feature_counts, path=['click', 'time_of_day'], values='count',
                 color='count', color_continuous_scale='Teal',
                 title='Ad Clicks by time_of_day')

fig.update_traces(
    textinfo='label+percent entry',  
    texttemplate='<b>%{label}</b><br>%{percentEntry:.2%}'
)

fig.update_layout(title_text='Ad Clicks by Time of Day ',
                  title_x=0.5, width=900, height=600)  

fig.show()


In [ ]:
feature_counts = df.groupby(['device_type', 'ad_position'])['click'].size().reset_index(name='click')
fig = px.bar(feature_counts,
             x='device_type', y='click', color='ad_position',
             title='Ad Clicks by Device Type and Ad Position',
             labels={"device_type": "Device Type", "click": "Number of Clicks", "ad_position": "Ad Position"},
             color_discrete_sequence=px.colors.qualitative.T10,
             text='click',
             barmode="group")
fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
fig.update_layout(
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    title_x=0.5,
    xaxis_title="Device Type",
    yaxis_title="Number of Clicks",
    legend_title="Ad Position",
    bargap=0.5,
    bargroupgap=0.1
)

fig.show()

In [ ]:
from dython.nominal import associations
associations_df = associations(df, nominal_columns='all', plot=False)

corr_matrix = associations_df['corr']

plt.figure(figsize=(20, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', linewidths=0.5)
plt.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

object_columns = df.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[(f'{col}_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [col]) for col in object_columns],
    remainder='passthrough'
)

In [ ]:
from sklearn.preprocessing import LabelEncoder
enco=LabelEncoder()
df['gender']=enco.fit_transform(df['gender'])

In [ ]:
x=df.drop(columns=['id','click'], axis=1)
y=df['click']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, random_state=3)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

model = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=94, use_label_encoder=False, eval_metric='logloss')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('adasyn', ADASYN(random_state=94)),
    ('classifier', model)
])

In [ ]:
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)

In [ ]:
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))